# P3 & P6 — best-epoch · MIMIC 3/6/10% + IU 3% + ablation  (an toan 12h)

Moi **Save Version** khai 1 danh sach **JOBS**. Moi job = 1 phuong phap (co the kem 1 ablation),
chay best-epoch (~1.5-2.5h/job). **GIU JOBS <= 4** -> chac chan duoi 12h.

- Chon-epoch GIONG HET Forget-MI (eval moi epoch tren D_t_final, do epoch gan GOLD nhat) -> CONG BANG.
- Eval-moi-epoch da lam NHE (bo cosine gold) nen nhanh + doan duoc thoi gian.


In [ ]:
# Cell 1: setup
import os, subprocess
WORK='/kaggle/working'; REPO=f'{WORK}/Forget-MI-LoKU'
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','https://github.com/nhnhu146/Forget-MI-LoKU.git',REPO],check=True)
else:
    subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
os.chdir(REPO)
assert os.path.exists('training/adv_common.py'),'push code truoc + re-import notebook'
subprocess.run(['pip','install','-q','pydicom','scikit-image','scikit-learn','pyyaml','wandb','seaborn==0.13.2'],check=True)
subprocess.run(['pip','install','-q','transformers==4.38.0','peft==0.10.0','accelerate==0.27.0'],check=True)
import torch; assert torch.cuda.is_available(),'Bat GPU'
print('Commit:',subprocess.check_output(['git','rev-parse','--short','HEAD'],text=True).strip())
print('GPU   :',torch.cuda.get_device_name(0))


In [ ]:
# Cell 2: CHON + JOBS (giu <=4 job / Save Version)
import glob, os

DATASET    = 'mimic'   # 'mimic' | 'iu'
FORGET_PCT = 3         # 3 | 6 | 10  (iu: 3)
SEED       = 42

EVAL_EVERY_EPOCH = True   # best-epoch (giong Forget-MI). ~1.5-2.5h/job. False -> chi `last` E30 (~0.4h).
RUN_EVAL_REF     = True    # eval OG + GOLD (moc vang). Chi can 1 LAN / dataset+% -> cac SV sau dat False.

# ---- JOBS cho Save Version NAY. m='p3'/'p6'; abl=None hoac ten ablation. GIU <= 4 dong ----
#   ablation: 'no_uumu' | 'no_fila'(chi p6) | 'no_noise' | 'p6_gate_free' | 'p6_gate_reg'
JOBS = [
    {'m':'p3'},        # P3 core
    {'m':'p6'},        # P6 core
]
# vi du ablation:  {'m':'p6','abl':'no_uumu'}, {'m':'p6','abl':'no_fila'}, {'m':'p6','abl':'no_noise'}

assert DATASET in ('mimic','iu') and FORGET_PCT in (3,6,10)
if DATASET=='iu': assert FORGET_PCT==3,'IU chi co 3%'
assert len(JOBS)<=5,'>5 job co the qua 12h — tach ra 2 Save Version'

def fd(*slugs):
    for s in slugs:
        if os.path.isdir(f'/kaggle/input/{s}'): return f'/kaggle/input/{s}'
        h=glob.glob(f'/kaggle/input/datasets/*/{s}')
        if h: return sorted(h)[0]
    return None
def bins(root): return sorted(glob.glob(os.path.join(root,'**','pytorch_model.bin'),recursive=True),key=len)

if DATASET=='mimic':
    DATA=fd('forget-mi-data'); MOD=fd('forget-mi-models-full','forget-mi-models')
    assert DATA and MOD,'Add forget-mi-data + forget-mi-models-full'
    BASE=os.path.dirname([b for b in bins(MOD) if 'training_original_model' in b][0])
    gold=[b for b in bins(MOD) if f'model_retrained_{FORGET_PCT}per' in b]
    GOLD=os.path.dirname(gold[0]) if gold else BASE
    TEXT=os.path.join(DATA,'data','metadata'); IMG=os.path.join(DATA,'data','img_data')
    SPLIT='./data_splits/mimic-cxr-sub-img-edema-split-manualtest.csv'
    FORGET=f'./data_splits/forget_set_{FORGET_PCT}per.csv'; tag=f'{FORGET_PCT}per'
else:
    DATA=fd('forget-mi-data-iu'); MOD=fd('forget-mi-models-iu'); MODRE=fd('forget-mi-models-iu-re')
    assert DATA and MOD and fd('chest-xrays-indiana-university'),\
        'Add forget-mi-data-iu + forget-mi-models-iu + forget-mi-models-iu-re + chest-xrays-indiana-university'
    ogb=[b for b in bins(MOD) if 'model_og' in b.lower() or 'base_model' in b.lower()] or bins(MOD)
    BASE=os.path.dirname(ogb[0])
    reb=(bins(MODRE) if MODRE else []) or [b for b in bins(MOD) if 'retrain' in b.lower()]
    GOLD=os.path.dirname(reb[0]) if reb else BASE
    TEXT=os.path.join(DATA,'data','metadata'); IMG=fd('chest-xrays-indiana-university')
    sp=glob.glob(os.path.join(DATA,'**','iu-split.csv'),recursive=True) or \
       glob.glob(os.path.join(DATA,'**','*iu*split*.csv'),recursive=True)
    fg=glob.glob(os.path.join(DATA,'**',f'forget_set_{FORGET_PCT}per_iu.csv'),recursive=True)
    assert sp and fg,f'Khong thay iu-split / forget_set_iu trong {DATA}'
    SPLIT=sp[0]; FORGET=fg[0]; tag=f'iu{FORGET_PCT}per'

for n,p in {'BASE':BASE,'TEXT':TEXT,'IMG':IMG,'SPLIT':SPLIT,'FORGET':FORGET}.items():
    assert p and os.path.exists(p),f'Missing {n}: {p}'

OUT=f'/kaggle/working/adv_{tag}_s{SEED}'; RESULTS=f'/kaggle/working/results_{tag}.csv'
def hist(rid):  return f'/kaggle/working/perepoch_{rid}.csv'
def thist(rid): return f'/kaggle/working/test_history_{rid}.csv'
COMMON={'forget_set_path':FORGET,'base_model_path':BASE,'bert_pretrained_dir':BASE,
        'retrained_model_path':GOLD,'text_data_dir':TEXT,'img_data_dir':IMG,
        'data_split_path':SPLIT,'output_dir':OUT,'results_csv_path':RESULTS,
        'eval_test_every_epoch':(1 if EVAL_EVERY_EPOCH else 0),'eval_test_light':1}
print('DATASET',DATASET,'| PCT',FORGET_PCT,'| SEED',SEED,'| tag',tag,'| JOBS',len(JOBS))
print('BASE',BASE); print('GOLD',GOLD); print('SPLIT',SPLIT); print('FORGET',FORGET)


In [ ]:
# Cell 3: chay JOBS (best-epoch). Uoc luong ~2.5h/job -> canh bao neu >4.
import os, subprocess, time
# gate_mode da bo: fusion gate LUON co dinh (LoKU chi toi uu LoRA) -> khong con ablation gate.
ABL={'no_uumu':{'ablate_uu_mu':1},'no_fila':{'loku_random_init':1},'no_noise':{'use_noise':0}}
SCRIPT={'p3':'training/forgetmi_p3.py','p6':'training/forgetmi_p6.py'}
LOG=[]
if len(JOBS)>4: print(f'!!  {len(JOBS)} job (~{len(JOBS)*2.5:.0f}h) — CO THE qua 12h, can nhac tach!')
def run(job):
    m=job['m']; abl=job.get('abl'); rid=f'{m}_{tag}_s{SEED}'+(f'_{abl}' if abl else '')
    ovr=dict(COMMON); ovr['id']=rid; ovr['history_csv_path']=hist(rid); ovr['test_history_csv_path']=thist(rid)
    if abl:
        assert abl in ABL,f'ablation la: {list(ABL)}'
        assert not (abl=='no_fila' and m=='p3'),f'{abl} chi cho p6'
        ovr.update(ABL[abl])
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled','PYTORCH_CUDA_ALLOC_CONF':'expandable_segments:True'}
    cmd=['python',SCRIPT[m],'--config','config_advanced_kaggle.yaml','--seed',str(SEED),'--fresh','--override',arg]
    print('='*70+f'\nJOB {rid}\n'+'='*70); t0=time.time()
    try: subprocess.run(cmd,env=env,check=True); LOG.append((rid,'OK',round((time.time()-t0)/3600,2)))
    except subprocess.CalledProcessError as e: print('FAIL',rid,e.returncode); LOG.append((rid,f'FAIL{e.returncode}',round((time.time()-t0)/3600,2)))
_all=time.time()
for j in JOBS: run(j)
print(f'\nTONG {(time.time()-_all)/3600:.2f}h:'); [print(' ',*x) for x in LOG]


In [ ]:
# Cell 4: eval OG + GOLD tren D_t_final (moc vang). Chi can 1 LAN / dataset+%.
import os, subprocess
def evalref(label, mpath):
    ovr=dict(COMMON); ovr['results_csv_path']=RESULTS
    arg=','.join(f'{k}={v}' for k,v in ovr.items())
    env={**os.environ,'PYTHONPATH':'.','WANDB_MODE':'disabled'}
    cmd=['python','training/forgetmi_eval_only.py','--config','config_advanced_kaggle.yaml','--seed',str(SEED),
         '--label',label,'--model_type','pretrained','--model_path',mpath,'--method','reference','--override',arg]
    print('eval-ref',label)
    try: subprocess.run(cmd,env=env,check=True)
    except subprocess.CalledProcessError as e: print('FAIL',label,e.returncode)
if RUN_EVAL_REF: evalref(f'og_{tag}',BASE); evalref(f're_{tag}',GOLD)
else: print('RUN_EVAL_REF=False -> bo qua (dung moc vang tu SV truoc)')


In [ ]:
# Cell 5: BANG KET QUA + do epoch tot nhat (gan GOLD nhat)
import os, glob, pandas as pd
pd.set_option('display.width',220); pd.set_option('display.max_columns',40)
if os.path.exists(RESULTS):
    df=pd.read_csv(RESULTS)
    c=[x for x in ['method','checkpoint_kind','id','Forget_AUC','Forget_Macro_F1','Test_AUC',
       'Test_Macro_F1','MIA','MIA_paper','forget_ce','test_ce','unlearn_core_hours'] if x in df.columns]
    print('===== RESULTS (last + selected + reference) ====='); print(df[c].to_string(index=False))
    g=df[(df.get('method')=='reference') & df['run_id'].astype(str).str.startswith('re')]
    gDf,gMIA,gDt=(float(g.iloc[-1]['Forget_AUC']),float(g.iloc[-1]['MIA']),float(g.iloc[-1]['Test_AUC'])) if len(g) else (0.5,0.42,0.62)
else:
    gDf,gMIA,gDt=0.5,0.42,0.62
print(f'\nGOLD: Df-AUC {gDf:.3f}  MIA {gMIA:.3f}  Dt-AUC {gDt:.3f}')

rows=[]
for f in sorted(glob.glob('/kaggle/working/test_history_*.csv')):
    pe=pd.read_csv(f); rid=os.path.basename(f)[len('test_history_'):-4]
    pe['score']=(pe['Df_AUC']-gDf).abs()+(pe['MIA']-gMIA).abs()+(gDt-pe['Dt_AUC']).clip(lower=0)
    b=pe.loc[pe['score'].idxmin()]
    print(f'\n=== {rid} ===')
    print(pe[['epoch','Df_AUC','Df_F1','Dt_AUC','Dt_F1','MIA','MIA_paper','forget_ce','test_ce']].to_string(index=False))
    print(f'>>> BEST = E{int(b.epoch)}  Df {b.Df_AUC:.3f}/{b.Df_F1:.3f}  Dt {b.Dt_AUC:.3f}/{b.Dt_F1:.3f}  MIA {b.MIA:.3f}')
    rows.append({'run':rid,'best_epoch':int(b.epoch),'Df_AUC':b.Df_AUC,'Df_F1':b.Df_F1,
                 'Dt_AUC':b.Dt_AUC,'Dt_F1':b.Dt_F1,'MIA':b.MIA,'MIA_paper':b.MIA_paper})
if rows:
    out=pd.DataFrame(rows); out.to_csv('/kaggle/working/best_epoch_summary.csv',index=False)
    print('\n===== BEST-EPOCH (moi run) ====='); print(out.to_string(index=False))
else:
    print('\n(EVAL_EVERY_EPOCH=False -> dung `last` E30 o bang RESULTS tren.)')
print('\nTAI VE: results_*.csv + test_history_*.csv + best_epoch_summary.csv')
